<a href="https://colab.research.google.com/github/Adamachmad/F1G123047/blob/main/F1G123047_KHALIFAH_ADAM_AHMAD_T3_ML_Prediksi_Rating_Anime_Regresi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [4]:
# load data anime.csv
df = pd.read_csv('/content/sample_data/anime.csv')

print("=== INFORMASI DATASET ===")
print(df.info())
display(df.head())

=== INFORMASI DATASET ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB
None


,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [5]:
# PREPROCESSING:
# 1. Membuang baris yang kosong
df_clean = df.dropna().copy()

# 2. Membuang baris yang episodenya 'Unknown' lalu mengubahnya jadi angka
df_clean = df_clean[df_clean['episodes'] != 'Unknown']
df_clean['episodes'] = df_clean['episodes'].astype(int)

# 3. Membuang kolom id dan nama yang tidak berpola matematis
df_clean = df_clean.drop(columns=['anime_id', 'name'])

# 4. One-Hot Encoding untuk 'type' (TV, Movie, OVA, dll) dan 'genre' [2]
type_encoded = pd.get_dummies(df_clean['type'], prefix='type', drop_first=True)
genre_encoded = df_clean['genre'].str.get_dummies(sep=', ')
df_clean = pd.concat([df_clean.drop(columns=['type', 'genre']), type_encoded, genre_encoded], axis=1)

# Menentukan Fitur (X) dan Target (y)
X = df_clean.drop('rating', axis=1)
y = df_clean['rating']

# Menyamakan Skala menggunakan Standard Scaler untuk kolom members dan episodes
scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[['episodes', 'members']] = scaler.fit_transform(X_scaled[['episodes', 'members']])

print("Data bersih dan siap untuk Regresi! Jumlah data:", len(df_clean))

Data bersih dan siap untuk Regresi! Jumlah data: 11830


In [6]:
test_sizes = [0.3, 0.2, 0.1]
# Menggunakan 3 Algoritma Regresi
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
}

hasil_evaluasi = []

print("=== MEMULAI EKSPERIMEN REGRESI & EVALUASI ===\n")

for size in test_sizes:
    train_pct = int((1 - size) * 100)
    test_pct = int(size * 100)
    print(f"--- RASIO SPLIT: {train_pct}% Training : {test_pct}% Testing ---")

    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=size, random_state=42)

    for nama_model, model in models.items():
        # Melatih mesin
        model.fit(X_train, y_train)

        # Tebakan mesin
        y_pred = model.predict(X_test)

        # Evaluasi Metrik Regresi
        r2 = r2_score(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)

        hasil_evaluasi.append({
            'Rasio Split': f"{train_pct}:{test_pct}",
            'Algoritma': nama_model,
            'R2 Score': r2,
            'MAE': mae,
            'RMSE': rmse
        })
        print(f"[{nama_model}] R2 Score: {r2:.4f} | MAE: {mae:.4f} | RMSE: {rmse:.4f}")
    print("\n")

=== MEMULAI EKSPERIMEN REGRESI & EVALUASI ===

--- RASIO SPLIT: 70% Training : 30% Testing ---
[Linear Regression] R2 Score: 0.3580 | MAE: 0.6300 | RMSE: 0.8174
[Decision Tree] R2 Score: 0.5150 | MAE: 0.5161 | RMSE: 0.7105
[Random Forest] R2 Score: 0.5976 | MAE: 0.4719 | RMSE: 0.6472


--- RASIO SPLIT: 80% Training : 20% Testing ---
[Linear Regression] R2 Score: 0.3709 | MAE: 0.6278 | RMSE: 0.8124
[Decision Tree] R2 Score: 0.5471 | MAE: 0.4998 | RMSE: 0.6893
[Random Forest] R2 Score: 0.6207 | MAE: 0.4626 | RMSE: 0.6309


--- RASIO SPLIT: 90% Training : 10% Testing ---
[Linear Regression] R2 Score: 0.3766 | MAE: 0.6236 | RMSE: 0.8154
[Decision Tree] R2 Score: 0.5290 | MAE: 0.5110 | RMSE: 0.7088
[Random Forest] R2 Score: 0.6197 | MAE: 0.4641 | RMSE: 0.6369




In [7]:
# Membuat tabel perbandingan performa
df_hasil = pd.DataFrame(hasil_evaluasi)

print("=== TABEL PERBANDINGAN PERFORMA MODEL REGRESI ===")
# Diurutkan dari R2 Score terbaik (Mendekati 1.0)
display(df_hasil.sort_values(by='R2 Score', ascending=False).reset_index(drop=True))

=== TABEL PERBANDINGAN PERFORMA MODEL REGRESI ===


,Rasio Split,Algoritma,R2 Score,MAE,RMSE
0,80:20,Random Forest,0.620690,0.462630,0.630858
1,90:10,Random Forest,0.619706,0.464071,0.636855
2,70:30,Random Forest,0.597588,0.471874,0.647172
3,80:20,Decision Tree,0.547150,0.499841,0.689305
4,90:10,Decision Tree,0.528960,0.510961,0.708777
5,70:30,Decision Tree,0.515007,0.516096,0.710479
6,90:10,Linear Regression,0.376650,0.623624,0.815356
7,80:20,Linear Regression,0.370939,0.627785,0.812420
8,70:30,Linear Regression,0.357972,0.629992,0.817449
